# Re-OCR via Mistral OCR API
Pipeline de re-OCRisation utilisant `mistral-ocr-latest` sur les images Gallica IIIF.

**Avantages vs pipeline GPU original :**
- Pas de GPU nécessaire
- 1 appel API par page (vs N appels par bloc)
- Parallélisable
- ~10-20× plus rapide

In [ ]:
# ── Cellule 1 : Dépendances ────────────────────────────────────────────────
!pip install mistralai lxml requests -q

In [ ]:
# ── Cellule 2 : Montage Drive ──────────────────────────────────────────────
import os, shutil

mountpoint = '/content/gdrive'   # on évite /content/drive qui pose problème
if os.path.isdir(mountpoint):
    os.system(f'fusermount -uz {mountpoint} 2>/dev/null || umount -l {mountpoint} 2>/dev/null')
    shutil.rmtree(mountpoint, ignore_errors=True)

from google.colab import drive
drive.mount(mountpoint)
print('Drive monté sur', mountpoint)

In [ ]:
# ── Cellule 3 : Configuration ──────────────────────────────────────────────
from google.colab import userdata

# Clé API Mistral (stockée dans les secrets Colab : Outils > Secrets)
# ou remplace par : MISTRAL_API_KEY = 'ta-clé-ici'
try:
    MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
except:
    MISTRAL_API_KEY = 'ta-clé-mistral-ici'  # ← remplace si pas de secrets Colab

# Dossier source : METS/ALTO originaux BnF (non perdus, à adapter)
# Structure attendue : SOURCE_DIR/<fascicule_id>_reocr/{toc/T*.xml, ocr/X*.xml, manifest.xml}
SOURCE_DIR  = '/content/gdrive/MyDrive/CHEMIN/VERS/TES/FASCICULES'  # ← à adapter

# Dossier de sortie pour les résultats re-OCR
OUTPUT_DIR  = '/content/gdrive/MyDrive/reocr_mistral_results'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Source  : {SOURCE_DIR}')
print(f'Sortie  : {OUTPUT_DIR}')

In [ ]:
# ── Cellule 4 : Fonctions utilitaires ─────────────────────────────────────
import re
import xml.etree.ElementTree as ET
from pathlib import Path

NS_ALTO = 'http://www.loc.gov/standards/alto/ns-v3#'
NS_METS = 'http://www.loc.gov/METS/'

def get_ark_from_manifest(manifest_path: Path) -> str:
    """Extrait l'ARK Gallica depuis manifest.xml."""
    tree = ET.parse(manifest_path)
    root = tree.getroot()
    # Cherche ark:/12148/bpt6k...
    text = ET.tostring(root, encoding='unicode')
    m = re.search(r'ark:/12148/(bpt6k[\w]+)', text)
    return m.group(1) if m else None

def iiif_url(ark: str, page: int, width: int = 2000) -> str:
    """URL IIIF Gallica pour une page donnée."""
    return (f'https://gallica.bnf.fr/iiif/ark:/12148/{ark}'
            f'/f{page}/full/{width},/0/native.jpg')

def get_alto_page_num(alto_filename: str) -> int:
    """X0000001.xml → 1"""
    m = re.search(r'X0*(\d+)', alto_filename)
    return int(m.group(1)) if m else 1

def parse_alto_blocks(alto_path: Path) -> tuple[dict, int, int]:
    """Retourne {block_id: {x,y,w,h,texte_original}}, largeur, hauteur."""
    tree = ET.parse(alto_path)
    root = tree.getroot()
    page = root.find(f'.//{{{NS_ALTO}}}Page')
    pw = int(page.get('WIDTH',  1)) if page is not None else 1
    ph = int(page.get('HEIGHT', 1)) if page is not None else 1
    blocks = {}
    for tb in root.iter(f'{{{NS_ALTO}}}TextBlock'):
        bid = tb.get('ID', '')
        lines = []
        for tl in tb.findall(f'{{{NS_ALTO}}}TextLine'):
            words = ' '.join(s.get('CONTENT','') for s in tl.findall(f'{{{NS_ALTO}}}String'))
            if words.strip(): lines.append(words.strip())
        blocks[bid] = {
            'x': int(tb.get('HPOS',0)), 'y': int(tb.get('VPOS',0)),
            'w': int(tb.get('WIDTH',0)), 'h': int(tb.get('HEIGHT',0)),
            'texte_original': ' '.join(lines),
        }
    return blocks, pw, ph

print('Fonctions chargées.')

In [ ]:
# ── Cellule 5 : Appel Mistral OCR ─────────────────────────────────────────
import base64, requests
from mistralai import Mistral

client = Mistral(api_key=MISTRAL_API_KEY)

def ocr_page_from_url(image_url: str) -> list[dict]:
    """
    Envoie une image (URL) à mistral-ocr-latest.
    Retourne une liste de blocs : [{text, bbox: {x,y,w,h}}]
    """
    resp = client.ocr.process(
        model='mistral-ocr-latest',
        document={'type': 'image_url', 'image_url': image_url},
        include_image_base64=False,
    )
    # resp.pages[0].blocks : liste de blocs avec texte et coordonnées
    blocks = []
    for page in resp.pages:
        for block in page.blocks:
            bbox = block.bbox  # {x_min, y_min, x_max, y_max} normalisé 0-1
            blocks.append({
                'text': block.text,
                'x_min': bbox.x_min, 'y_min': bbox.y_min,
                'x_max': bbox.x_max, 'y_max': bbox.y_max,
            })
    return blocks


def match_mistral_to_alto(mistral_blocks: list[dict],
                           alto_blocks: dict, pw: int, ph: int) -> dict:
    """
    Associe chaque bloc ALTO au bloc Mistral le plus proche (IoU).
    Retourne {alto_block_id: texte_mistral}
    """
    def to_norm(b, pw, ph):
        """Convertit coordonnées ALTO (pixels) en normalisé 0-1."""
        return {
            'x_min': b['x'] / pw,
            'y_min': b['y'] / ph,
            'x_max': (b['x'] + b['w']) / pw,
            'y_max': (b['y'] + b['h']) / ph,
        }

    def iou(a, b):
        ix1 = max(a['x_min'], b['x_min'])
        iy1 = max(a['y_min'], b['y_min'])
        ix2 = min(a['x_max'], b['x_max'])
        iy2 = min(a['y_max'], b['y_max'])
        inter = max(0, ix2-ix1) * max(0, iy2-iy1)
        area_a = (a['x_max']-a['x_min']) * (a['y_max']-a['y_min'])
        area_b = (b['x_max']-b['x_min']) * (b['y_max']-b['y_min'])
        union  = area_a + area_b - inter
        return inter / union if union else 0

    result = {}
    for bid, bdata in alto_blocks.items():
        alto_norm = to_norm(bdata, pw, ph)
        best_iou, best_text = 0, bdata['texte_original']  # fallback = texte original
        for mb in mistral_blocks:
            score = iou(alto_norm, mb)
            if score > best_iou:
                best_iou, best_text = score, mb['text']
        result[bid] = {'texte': best_text, 'iou': round(best_iou, 3)}
    return result

print('Fonctions OCR chargées.')

In [ ]:
# ── Cellule 6 : Sauvegarde résultats ──────────────────────────────────────
import json

def save_results(fascicule_id: str, page_num: int,
                 matched: dict, output_dir: str):
    """
    Sauvegarde les textes re-OCRisés dans un JSON structuré.
    Format : {block_id: {texte, iou}}
    Compatible avec le pipeline de post-traitement existant.
    """
    fasc_dir = Path(output_dir) / f'{fascicule_id}_reocr'
    fasc_dir.mkdir(parents=True, exist_ok=True)
    out_file = fasc_dir / f'X{page_num:07d}_mistral.json'
    out_file.write_text(
        json.dumps(matched, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    return out_file

print('Fonction de sauvegarde chargée.')

In [ ]:
# ── Cellule 7 : Pipeline principal ─────────────────────────────────────────
import time
from pathlib import Path

source_path = Path(SOURCE_DIR)
folders = sorted(source_path.glob('*_reocr'))
print(f'{len(folders)} fascicules trouvés\n')

all_results = []
IIIF_WIDTH  = 2000   # résolution image envoyée à Mistral (pixels)
DELAY_SEC   = 1.0    # pause entre appels API (rate limit)

for folder in folders:
    fascicule_id = re.sub(r'\D', '', folder.name)
    print(f'\n══ {fascicule_id} ══')

    # ARK Gallica
    manifest = folder / 'manifest.xml'
    if not manifest.exists():
        print('  ⚠ manifest.xml absent, ignoré'); continue
    ark = get_ark_from_manifest(manifest)
    if not ark:
        print('  ⚠ ARK introuvable, ignoré'); continue
    print(f'  ARK : {ark}')

    # Pages ALTO
    ocr_dir = folder / 'ocr'
    alto_files = sorted(ocr_dir.glob('X*.xml'))
    print(f'  {len(alto_files)} page(s) à traiter')

    for alto_path in alto_files:
        page_num = get_alto_page_num(alto_path.name)

        # Vérifier si déjà traité (reprise sur erreur)
        out_check = Path(OUTPUT_DIR) / f'{fascicule_id}_reocr' / f'X{page_num:07d}_mistral.json'
        if out_check.exists():
            print(f'  p{page_num} ✓ déjà traité')
            continue

        try:
            # 1. Charger les blocs ALTO
            alto_blocks, pw, ph = parse_alto_blocks(alto_path)

            # 2. URL image Gallica
            img_url = iiif_url(ark, page_num, IIIF_WIDTH)

            # 3. Appel Mistral OCR
            mistral_blocks = ocr_page_from_url(img_url)

            # 4. Alignement avec les blocs ALTO
            matched = match_mistral_to_alto(mistral_blocks, alto_blocks, pw, ph)

            # 5. Sauvegarde
            out_file = save_results(fascicule_id, page_num, matched, OUTPUT_DIR)

            n_good = sum(1 for v in matched.values() if v['iou'] > 0.3)
            print(f'  p{page_num} ✓ {len(mistral_blocks)} blocs Mistral → '
                  f'{n_good}/{len(matched)} alignés (IoU>0.3) → {out_file.name}')

            all_results.append({
                'fascicule': fascicule_id, 'page': page_num,
                'n_blocs_alto': len(alto_blocks),
                'n_blocs_mistral': len(mistral_blocks),
                'n_alignes': n_good,
            })

        except Exception as e:
            print(f'  p{page_num} ❌ {e}')

        time.sleep(DELAY_SEC)

print(f'\n✅ Terminé — {len(all_results)} pages traitées')

In [ ]:
# ── Cellule 8 : Rapport de synthèse ───────────────────────────────────────
import json
from pathlib import Path

rapport = {
    'total_pages': len(all_results),
    'total_fascicules': len({r['fascicule'] for r in all_results}),
    'pages': all_results,
}
rapport_path = Path(OUTPUT_DIR) / 'rapport_mistral_ocr.json'
rapport_path.write_text(json.dumps(rapport, ensure_ascii=False, indent=2))

print(f'Fascicules traités : {rapport["total_fascicules"]}')
print(f'Pages traitées     : {rapport["total_pages"]}')
if all_results:
    avg = sum(r['n_alignes']/max(r['n_blocs_alto'],1) for r in all_results) / len(all_results)
    print(f'Taux alignement    : {avg*100:.0f}% (IoU>0.3)')
print(f'Rapport            : {rapport_path}')